In [ ]:
## Packages Import
%matplotlib widget
import copy
import sys
import numpy             as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from matplotlib.ticker import MaxNLocator

sys.path.append("./src/")
from VLMSurface import VLMSurface
from VLMSolver import VLMSolver
from Utilities import *



plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['savefig.format'] = 'pdf'  # Put PDF as default save format
plt.rcParams['savefig.bbox'] = 'tight'  # Cut the figure to the content

In [ ]:
## Parameters Definition

# Wing geometry
B =1   # wing span                 [m]
AR = 1       # aspect ratio             [-]
ALPHA  = 0     # pitch angle           [deg] - positive definite for counterclockwise rotations about y-axis
BETA   = 10    # leeway angle               [deg] - positive definite for counterclockwise rotations about z-axis
LAMBDA = 0      # middle-chord sweep angle  [deg] - positive definite for counterclockwise rotations about x-axis  Y-AXIS
DELTA  = 0      # dihedral angle            [deg] - positive definite for rotations oriented towards positive y-axis  X-AXIS, négatif pour dyhedre classique 
PHI    = 0      # twist tip angle           [deg] - negative for washout

SYM   = True          # symmetric wing configuration
SPACE = True  # spacing distribution, False = uniform, True = cos
SHAPE = "rectangular"   # shape of the wing
TIME  = "classic"       # time step distribution - classic or cosine at the starting vortex
FREE  = "antisymmetric0"     # free surface boundary condition - symmetric (wall) ; antisymmetric (zero delta pressure) ; None

FROUDE = 1.5

# Flow properties
U = 1#FROUDE*np.sqrt(B*9.81)     # inflow velocity [m/s]

# Wing discretization
N = 40   # number of panels in spanwise direction
M = 10    # number of panels in chordwise direction

# Time simulation parameters
T  =  8*B/U/AR # length of the simulation in s
DT = T/50      # time step lentgh 




In [ ]:
surface = VLMSurface(origin=np.array([0,0,0]),
                            plan=0,
                            boundary=FREE,
                            shedding="both",
                            b=B,
                            c=chord_fn((N+1), AR, B,False, SPACE, "rectangular"),
                            alpha=np.deg2rad(ALPHA),
                            beta=np.deg2rad(BETA),
                            lamb=np.deg2rad(LAMBDA),
                            delta=np.deg2rad(DELTA),
                            phi=np.deg2rad(PHI),
                            sym=False,
                            space=SPACE,
                            n=N,
                            m=M,
                            vent=True)

surface1 = VLMSurface(origin=np.array([0,0,B]),
                            plan=1,
                            boundary=FREE,
                            shedding="both0",
                            b=B,
                            c=chord_fn((N+1), AR, B,SYM, SPACE, "elliptical"),
                            alpha=np.deg2rad(ALPHA),
                            beta=np.deg2rad(BETA),
                            lamb=np.deg2rad(LAMBDA),
                            delta=np.deg2rad(DELTA),
                            phi=np.deg2rad(PHI),
                            sym=SYM,
                            space=SPACE,
                            n=N,
                            m=M,
                            vent=True)
# plot2D([surface],False,1)

vlm = VLMSolver([surface],np.array([U,0,0]),boundary=FREE)
vlm._time_sim()
vlm._kuttas_loads()
plt.show()
plot3D(vlm.surfaces)
#vlm._ventilation()
#plot_ventilation(surface, True)
# print("rc : ", vlm.rc)
#S = np.sum(np.array([p.area for p in surface.wing_panels["real"]]).reshape(M,2*N), axis=0)
print("Cl = ",2*surface.loads[1]/(B**2/AR))
print("Cl = ", surface.CL)
#print("Cl = ",2*surface1.loads[1]/(B**2/AR))

# plot3D(vlm.surfaces)
surface.plot_Cl_2d()
surface.plot_load()
surface.plot_gamma()
# surface.plot_area()


In [ ]:
def L_foils(fly_height=0.0, shape="L", foil="both"):
    N = 40
    M = 10
    B = 1.13
    AR = B/0.16
    U = 15*1.852/3.6
    SPACE = False
    FREE = "antisymmetric"
    root, tip, radius, center, H, AL, BE = {
        "L": (10, 90, 0.1, 0.65, [0,.1,.2,.3,.4,.5], [6.11,6.09,6.13,6.19,6.28,6.55], [1.24,1.52,1.98,2.79,4.46,8.80]),
        "V": (15, 45, 0.3, 0.5, [0,.1,.2,.3,.4,.5,.6], [9.10,9.36,9.55,9.73,9.94,10.44,12.06], [1.03,1.24,1.56,2.10,3.02,4.53,6.98]),
        "C": (20, 60, 0.5, 0.5, [0,.1,.2,.3,.4,.5,.6], [7.10,7.31,7.46,7.58,7.79,8.33,9.59], [1.12,1.36,1.76,2.47,3.64,5.82,10.87]),
        "J": (10, 75, 0.3, 0.7, [0,.1,.2,.3,.4,.5,.6], [6.89,6.95,7.01,7.06,7.15,7.37,8.30], [1.13,1.35,1.75,2.41,3.72,6.76,16.38])}[shape]
    ALPHA = float(np.interp(fly_height, H, AL))
    BETA = float(np.interp(fly_height, H, BE))
    surface3 = VLMSurface(origin=np.array([0,-1,-fly_height]),
                            plan=0,
                            boundary=FREE,
                            shedding="right",
                            b=B,
                            c=chord_fn((N+1), AR, B, False, SPACE, SHAPE),
                            alpha=np.deg2rad(ALPHA),
                            beta=np.deg2rad(BETA),
                            lamb=np.deg2rad(LAMBDA),
                            delta=np.deg2rad(DELTA),
                            phi=np.deg2rad(PHI),
                            sym=False,
                            space=SPACE,
                            n=N,
                            m=M,
                            vent=True,
                            root_ang=-root,
                            tip_ang=tip,
                            radius=radius,
                            center=center,
                            fly_height=fly_height)

    surface4 = VLMSurface(origin=np.array([0,1,-fly_height]),
                            plan=0,
                            boundary=FREE,
                            shedding="right",
                            b=B,
                            c=chord_fn((N+1), AR, B, False, SPACE, SHAPE),
                            alpha=np.deg2rad(ALPHA),
                            beta=np.deg2rad(BETA),
                            lamb=np.deg2rad(LAMBDA),
                            delta=np.deg2rad(DELTA),
                            phi=np.deg2rad(PHI),
                            sym=False,
                            space=SPACE,
                            n=N,
                            m=M,
                            vent=True,
                            root_ang=root,
                            tip_ang=-tip,
                            radius=radius,
                            center=center,
                            fly_height=fly_height)
    if foil=="port":
        vlm = VLMSolver([surface3],np.array([U,0,0]),FREE)
    elif foil=="starboard":
        vlm = VLMSolver([surface4],np.array([U,0,0]),FREE)
    else :
        vlm = VLMSolver([surface3, surface4],np.array([U,0,0]),FREE)
    return vlm

In [ ]:
vlm = L_foils(shape="C")
vlm._time_sim()
vlm._kuttas_loads()
plot3D(vlm.surfaces)
L_tot = 1025/9.81*np.sum([surface.loads for surface in vlm.surfaces], axis=0)
print("horizontal lift : ",L_tot[1])
print("vertical lift   : ",-L_tot[2])


In [ ]:
shapes          = ["L","C","J","V"]
heights         = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
Horizontal_load = {}
Vertical_load   = {}
for i,shape in enumerate(shapes) :
    h_p = [] 
    v_p = []
    h_s = []
    v_s = []
    for h in heights :
        vlm = L_foils(fly_height=h, shape=shape, foil="port")
        vlm._ventilation()
        L_tot = 1025*np.sum([surface.loads for surface in vlm.surfaces], axis=0)/9.81
        h_p.append(L_tot[1])
        v_p.append(-L_tot[2])
        vlm = L_foils(fly_height=h, shape=shape, foil="starboard")
        #vlm._ventilation()
        vlm._time_sim()
        vlm._kuttas_loads()
        L_tot = 1025*np.sum([surface.loads for surface in vlm.surfaces], axis=0)/9.81
        h_s.append(L_tot[1])
        v_s.append(-L_tot[2])
        Horizontal_load[shape] = [h_p, h_s]
        Vertical_load[shape]   = [v_p, v_s]
    print("Shape "+shape+" done")
    print("Horizontal_load = ",Horizontal_load)
    print("Vertical_load   = ",Vertical_load)
    if i == 0 :
        heights.append(0.6)
print("Horizontal_ref = ",Horizontal_load)
print("Vertical_ref   = ",Vertical_load)


In [ ]:
## test Global ventilation
surface3 = VLMSurface(origin=np.array([0,0,0.5*B]),
                        plan=0,
                        boundary=FREE,
                        shedding="right",
                        b=B,
                        c=chord_fn((N+1), AR, B, SYM, SPACE, SHAPE),
                        alpha=np.deg2rad(ALPHA),
                        beta=np.deg2rad(BETA),
                        lamb=np.deg2rad(LAMBDA),
                        delta=np.deg2rad(DELTA),
                        phi=np.deg2rad(PHI),
                        sym=SYM,
                        space=SPACE,
                        n=N,
                        m=M,
                        vent=True)
vlm = VLMSolver([surface3], np.array([U,0,0]),FREE)
vlm._ventilation()
print("Cl = ", 2*np.linalg.norm(surface3.loads[1:])/(surface3.area*U**2))
plot3D([surface3])
plot_ventilation(surface3)


In [ ]:
## Ventilation V4



def iterative_vent(beta,plot):
    BETA = beta
    surface3 = VLMSurface(origin=np.array([0,0,0.5*B]),
                        plan=0,
                        boundary=FREE,
                        shedding="right",
                        b=B,
                        c=chord_fn((N+1), AR, B, SYM, SPACE, SHAPE),
                        alpha=np.deg2rad(ALPHA),
                        beta=np.deg2rad(BETA),
                        lamb=np.deg2rad(LAMBDA),
                        delta=np.deg2rad(DELTA),
                        phi=np.deg2rad(PHI),
                        sym=SYM,
                        space=SPACE,
                        n=N,
                        m=M,
                        vent=True)
    da = np.zeros(2*N)
    vlm = VLMSolver([surface3], np.array([U,0,0]), FREE)
    vlm._time_sim(t=B*T/(AR*0.0453), dt=B*DT/(AR*0.0453), distribution=TIME)
    #plot3D([surface3],False)
    vlm._kuttas_loads()
    Cl = copy.copy(surface3.Cl_2d)
    print("Cl :",Cl)
    slope1 = 2*surface3.loads[1]*AR/(B**2)/(U**2)/np.deg2rad(BETA)
    print("slope",slope1)
    slope = copy.copy(surface3.Cl_2d)/np.deg2rad(BETA)
    a0 = 2*np.pi*np.ones_like(Cl)
    phi = np.zeros(2*N+1)
    k = 00
    aoa = np.pi*np.ones(2*N)            # just for the fisrst test
    while k<400 and np.linalg.norm(a0*(aoa) - Cl)>1e-2:          
        k += 1
        aoa = (Cl/(2*np.pi))-da
        surface3._ventilation_coef(aoa,1)
        a0 = copy.copy(surface3.ventilation["a0"])
        delta_aoa = 0.1*(a0*(aoa) - Cl)/(slope1)
        da = da + delta_aoa
        #phi = phi + np.concatenate([np.zeros(1),(delta_aoa[1:] + delta_aoa[:-1])/2, 0.5*np.array([delta_aoa[-1]])])
        surface3 = VLMSurface(origin=np.array([0,0,0.5*B]),
                        plan=0,
                        boundary=FREE,
                        shedding="right",
                        b=B,
                        c=chord_fn((N+1), AR, B, SYM, SPACE, SHAPE),
                        alpha=np.deg2rad(ALPHA),
                        beta=np.deg2rad(BETA),
                        lamb=np.deg2rad(LAMBDA),
                        delta=np.deg2rad(DELTA),
                        phi=phi,
                        sym=SYM,
                        space=SPACE,
                        n=N,
                        m=M,
                        vent=True)
        vlm = VLMSolver([surface3], np.array([U,0,0]), FREE)
        vlm._correct([da])
        vlm._time_sim(t=B*T/(AR*0.0453), dt=B*DT/(AR*0.0453), distribution=TIME)
        vlm._kuttas_loads()
        if k == 1:
            print("Correction [°]:", da) 
            print("a0 :", a0)
            print("normals :", vlm.normals[:20])
        Cl = copy.copy(surface3.Cl_2d)
    if k==400:
        print("Did not converge")
    if plot :
        surface3._ventilation_coef(aoa,1)
        plot3D([surface3],False)
        plot_ventilation(surface3)
    print("T  :",B*T/(AR*0.0453))
    print("DT :",B*DT/(AR*0.0453))
    return 2*surface3.loads*AR/(B**2)/(U**2)

print("Cl : ", iterative_vent(BETA,True)[1])







In [ ]:
# random test
N=6
M=3
surface = VLMSurface(origin=np.array([0,0,0]),
                        plan=0,
                        boundary=FREE,
                        shedding="both",
                        b=B,
                        c=chord_fn((N+1), 4, B, False, SPACE, SHAPE),
                        alpha=-np.deg2rad(0),
                        beta=np.deg2rad(10),
                        lamb=np.deg2rad(LAMBDA),
                        delta=np.deg2rad(DELTA),
                        phi=np.deg2rad(PHI),
                        sym=False,
                        space=SPACE,
                        n=N,
                        m=M,
                        vent=False)
surface._set_normals(0)
orientation=(1)                                                 # orientation of the normals in comparison to the flow / 1 for upward -1 for downward
mean_normal = np.mean(surface.normals, axis=0)
mean_normal_unit = mean_normal / np.linalg.norm(mean_normal)
cos_avg = mean_normal_unit[0] 
avg_aoa = np.abs(np.arcsin(cos_avg))            # average angle of attack
print("original aoa",np.rad2deg(avg_aoa))
if cos_avg < 0 :
    orientation = -1
surface.orientation = orientation
correction2 = np.linspace(np.deg2rad(0), np.deg2rad(-60),N)
vlm = VLMSolver([surface], np.array([U,0,0]), FREE)
vlm._correct([correction2])
# vlm._time_sim(T, DT, TIME)
# vlm._kuttas_loads()
plot3D([surface], False)
mean_normal = np.mean(surface.normals, axis=0)
mean_normal_unit = mean_normal / np.linalg.norm(mean_normal)
cos_avg = mean_normal_unit[0] 
avg_aoa = np.abs(np.arcsin(cos_avg))            # average angle of attack
print("estimated aoa",np.rad2deg(avg_aoa))
#print("CL           ",2*np.linalg.norm(surface.loads[1:])/(surface.area*U**2) )
# fig = plt.figure(figsize=(14/1.5, 8/1.5),constrained_layout=True)
# ax  = fig.add_subplot()
# ctrl     = np.array([p.ctr for p in surface.wing_panels["real"]])
# z        = np.sum(ctrl.reshape((M,2*N,3)), axis=0)[:,2]/M 
# ax.plot(z, surface.Cl_2d, label="new")
# Cl_old= [0.38032247, 0.36651316, 0.36249562, 0.36687121, 0.37359898, 0.38044368, 0.38645063, 0.39117703, 0.39440904, 0.39604532, 0.39604532, 0.3944090, 0.39117703 ,0.38645063, 0.38044368, 0.37359898, 0.36687121 ,0.36249562, 0.36651316, 0.38032247]
# ax.plot(z, Cl_old, "--", label="target")
# ax.plot(z, surface.Cl_old, ".-", label="old")
# ax.set_xlabel("Spanwise location "+ r"$y/b$")
# ax.set_ylabel("2D lift coefficient "+ r"$C_l$")
# ax.set_ylim(np.min(np.concatenate([surface.Cl_2d, np.array([0])])), np.max(surface.Cl_2d)*1.1)
# ax.legend()
# ax.grid()
# print("VLM lift coef           :", 2*surface.loads[1]*AR/(B**2), "(AR=",AR,"; ALPHA=",ALPHA,"°)" )

In [ ]:
## Class A simulation


def classA_sim(both, fly_regime, phi, plot):
    """ 
    Input :
        both       -> True if the two foils are simulated, False if only one the port one is simulated
        fly_regime -> Index of the flying height regime (equal to 10*flying height)
        phi        -> Array of correction angles for ventilation for each board, in rad
        plot       -> True if the results should be plotted
    Output :
        vlm        -> VLMSolver object containing the results of the simulation
        h          -> Total immersioon depth 
    """
    flying_height   = np.array([0, 0.1, 0.2, 0.3, 0.4, 0.5])
    Horizontal_lift = np.array([61, 63, 65, 68, 72, 79])
    AoA             = np.array([6.8, 6.87, 6.92, 6.75, 6.85, 7.12])        
    Leeway_angle    = np.array([1.22, 1.53, 2.02, 2.86, 4.55, 9.2])
    BETA            = Leeway_angle[fly_regime]
    ALPHA           = AoA[fly_regime]
    B_MAST     = 1.46
    FLYING_HEIGHT = flying_height[fly_regime]
    DELTA = -10.3
    LAMBDA = 0      # middle-chord sweep angle  [deg] - positive definite for counterclockwise rotations about x-axis  Y-AXIS
    PHI    = 0      # twist tip angle           [deg] - negative for washout

    SPACE = True  # spacing distribution, False = uniform, True = cos
    TIME  = "classic"       # time step distribution - classic or cosine at the starting vortex
    FREE  = "antisymmetric"     # free surface boundary condition - symmetric (wall) ; antisymmetric (zero delta pressure) ; None
    
    # Flow properties
    U = 15*1.852/3.6     # inflow velocity [m/s]

    # Wing discretization
    N = 10    # number of panels in spanwise direction
    M = 10    # number of panels in chordwise direction

    # Time simulation parameters
    T  =  15*0.16/U # length of the simulation in s
    DT = T/80      # time step lentgh 



    surface2 = VLMSurface(origin=np.array([0,-1,0]),
                            plan=0,
                            boundary=FREE,
                            shedding="none",
                            b=B_MAST-2*FLYING_HEIGHT/np.cos(np.deg2rad(DELTA)),
                            c=chord_fn((N+1), B_MAST/0.16, B_MAST, False, SPACE, "rectangular"),
                            alpha=np.deg2rad(ALPHA),
                            beta=np.deg2rad(BETA),
                            lamb=np.deg2rad(LAMBDA),
                            delta=np.deg2rad(DELTA),
                            phi=phi[0],
                            sym=False,
                            space=SPACE,
                            n=N,
                            m=M,
                            vent=True)
    B = 0.8
    surface3 = VLMSurface(origin=np.array([0,-1+B/2 + (B_MAST-2*FLYING_HEIGHT/np.cos(np.deg2rad(DELTA)))*np.sin(np.deg2rad(DELTA))/2, ((B_MAST*np.cos(np.deg2rad(DELTA))-2*FLYING_HEIGHT))/2]),
                            plan=1,
                            boundary=FREE,
                            shedding="left",
                            b=B,
                            c=chord_fn((N+1), B/0.16, B, False, SPACE, "rectangular"),
                            alpha=np.deg2rad(ALPHA),
                            beta=np.deg2rad(BETA),
                            lamb=np.deg2rad(LAMBDA),
                            delta=np.deg2rad(0),
                            phi=-phi[1],
                            sym=False,
                            space=SPACE,
                            n=N,
                            m=M,
                            vent=True)

    surface4 = VLMSurface(origin=np.array([0,1,0]),
                            plan=0,
                            boundary=FREE,
                            shedding="none",
                            b=B_MAST-2*FLYING_HEIGHT/np.cos(np.deg2rad(DELTA)),
                            c=chord_fn((N+1), B_MAST/0.16, B_MAST, False, SPACE, "rectangular"),
                            alpha=np.deg2rad(ALPHA),
                            beta=np.deg2rad(BETA),
                            lamb=np.deg2rad(LAMBDA),
                            delta=np.deg2rad(-DELTA),
                            phi=phi[0],
                            sym=False,
                            space=SPACE,
                            n=N,
                            m=M,
                            vent=True)

    surface5 = VLMSurface(origin=np.array([0, 1-(B_MAST-2*FLYING_HEIGHT/np.cos(np.deg2rad(DELTA)))*np.sin(np.deg2rad(DELTA))/2, ((B_MAST*np.cos(np.deg2rad(DELTA))-2*FLYING_HEIGHT))/2]),
                            plan=1,
                            boundary=FREE,
                            shedding="right",
                            b=B,
                            c=chord_fn((N+1), B/0.16, B, False, SPACE, "rectangular"),
                            alpha=np.deg2rad(ALPHA),
                            beta=np.deg2rad(BETA),
                            lamb=np.deg2rad(LAMBDA),
                            delta=np.deg2rad(0),
                            phi=-phi[1],
                            sym=False,
                            space=SPACE,
                            n=N,
                            m=M, 
                            vent=True)

    if both :
        surfaces = [surface2, surface3, surface4, surface5]
    else :
        surfaces = [surface4, surface5]
    if plot :
        plot2D(surfaces,False, 0)
    vlm = VLMSolver(surfaces, np.array([U,0,0]), FREE)
    # vlm._time_sim(t=T, dt=DT, distribution=TIME)
    # vlm._kuttas_loads()
    #L_tot = 1025*np.sum([surface.loads for surface in surfaces], axis=0)/9.81
    
    # print("Horizontal Lift : ", L_tot[1])
    # print("Vertical Lift : ", -L_tot[2]) 
    if plot : 
        
        plot3D(surfaces,False)
    
    return vlm, ((B_MAST*np.cos(np.deg2rad(DELTA))-2*FLYING_HEIGHT))/2



# vlm = classA_sim(both=False, fly_regime=4, phi=np.array([0,0,0,0]), plot=True)[0]
# print("Cl = ",vlm.surfaces[0].Cl_2d)
#L_tot = classA_vent(fly_regime=4, ventilate=2, plot=True)
# vent = [1, 2]
# flyght_reg = [0, 1, 2, 3, 4, 5]
# F_vert=[]
# F_hor=[]
# for f in flyght_reg :
#     vlm = classA_sim(both=True, fly_regime=f, phi=np.array([0,0,0,0]), plot=True)[0]
#     L_tot = 1025*np.sum([surface.loads for surface in vlm.surfaces], axis=0)/9.81
#     F_vert.append(-L_tot[2])
#     F_hor.append(L_tot[1])
# print("Vertical Lift = ", F_vert)
# print("Horizontal Lift = ", F_hor)

In [ ]:

vlm = classA_sim(True,0,np.array([0,0,0,0]),True)[0]
vlm._time_sim()
vlm._kuttas_loads()
#vlm._ventilation()
#plot3D(vlm.surfaces)
L_tot = 1025*np.sum([surface.loads for surface in vlm.surfaces], axis=0)/9.81
print("Vertical Lift = ", -L_tot[2])
print("Horizontal Lift = ", L_tot[1])

In [ ]:
## Ventilation class A

def classA_vent(fly_regime, ventilate, plot):
    da = np.tile(np.zeros(N), (ventilate,1))
    vlm, h =classA_sim(False, fly_regime, np.zeros(N+1), False)
    FROUDE = U/(np.sqrt(9.81*h))
    Cl = np.array([surface.Cl_2d for surface in vlm.surfaces])
    n_surf = ventilate
    Cl = np.abs(Cl[:n_surf])
    slope1 = np.abs(np.array([2*surface.loads[surface.plan+1]*AR/(B**2)/(U**2)/np.deg2rad(BETA) for surface in vlm.surfaces]))
    a0 = 2*np.pi*np.ones_like(Cl)
    delta_aoa = copy.copy(a0)
    phi = np.zeros((4, N+1))
    k = 00
    aoa = np.tile(np.pi*np.ones(N),(n_surf,1))            # just for the fisrst test
    while k<400 and np.max(np.abs(a0*aoa - Cl)) > 1e-2:          
        k += 1
        for i in range(ventilate):
            surface = vlm.surfaces[i]
            aoa[i] = (Cl[i]/(2*np.pi))-da[i]
            surface._ventilation_coef(aoa[i],FROUDE)
            a0[i] = copy.copy(surface.ventilation["a0"])
            delta_aoa[i] = 1000*0.05*(a0[i]*(aoa[i]) - Cl[i])/(slope1[i])
            da[i] = da[i] + delta_aoa[i]
            phi[i] = phi[i] + np.concatenate([np.zeros(1),(delta_aoa[i][1:] + delta_aoa[i][:-1])/2, 0*np.array([delta_aoa[i][-1]])])
        vlm = classA_sim(False, fly_regime, phi, False)[0]
        for i in range(ventilate):
            Cl[i] = np.abs(copy.copy(vlm.surfaces[i].Cl_2d))
    if k==400:
        print("Did not converge")
    if plot :
        plot3D(vlm.surfaces,False)
    return 1025*np.sum([surface.loads for surface in vlm.surfaces], axis=0)/9.81



In [ ]:
## Ventilated validation

beta = np.array([2.5,5,7.5,10,12.5,15,17.5,20,22.5,25,27.5,30])
asp = [0.5, 1, 1.5]
Fn  = [[2.5, 4.5],[1.5, 2.5, 3.5],[1.5, 2, 3]]
print("alpha = ", beta)
for i, ar in enumerate(asp) :
    AR = ar
    print("Ar : ", ar)
    for f in Fn[i] :
        FROUDE = f
        B      = 1/(9.81*FROUDE**2)
        cl_kut = []
        cd_kut = []
        for b in beta : 
            c = iterative_vent(b, False)
            cl_kut.append(c[1])
            cd_kut.append(c[0])
        print("Fn : ", f)
        print("cl_kut = ", cl_kut)
        print("cd_kut = ", cd_kut)
        print(" ")

In [ ]:
## Wetted validation

beta = np.linspace(0,12,13)
FREE = "no"
asp = [ 3]
Fn  = [ 1.5]
print("alpha = ", beta)
for ar in asp :
    AR = ar
    print("Ar : ", ar)
    for f in Fn :
        FROUDE = f
        B      = 1/(9.81*FROUDE**2)
        cl_kut = []
        cd_kut = []
        for b in beta : 
            surface3 = VLMSurface(origin=np.array([0,0,0.5*B]),
                        plan=0,
                        boundary=FREE,
                        shedding="both",
                        b=B,
                        c=chord_fn((N+1), AR, B, SYM, SPACE, SHAPE),
                        alpha=np.deg2rad(ALPHA),
                        beta=np.deg2rad(b),
                        lamb=np.deg2rad(LAMBDA),
                        delta=np.deg2rad(DELTA),
                        phi=np.deg2rad(PHI),
                        sym=SYM,
                        space=SPACE,
                        n=N,
                        m=M)
            vlm = VLMSolver([surface3], np.array([U,0,0]), FREE, ventilation=True, correction=np.zeros(2*N))
            vlm._time_sim(t=B*T/(AR*0.0453), dt=B*DT/(AR*0.0453), distribution=TIME)
            vlm._kuttas_loads()
            c = surface3.loads
            cl_kut.append(2*c[1]*AR/(B**2))
            cd_kut.append(2*c[0]*AR/(B**2))
        print("Fn : ", f)
        print("cl_kut = ", cl_kut)
        print("cd_kut = ", cd_kut)
        print(" ")